# Reducing H5 filesize
Removes high frequency force traces from H5 files to reduce overall file size.  Processed files are saved to a mirrored directory structure.  Any non-H5 files are copied as is.

In [5]:
# Importing libraries
from lumicks import pylake

import os
import shutil

In [6]:
# Parameters
input_path = "./examples/input/" # Path to folder containing H5s to process
output_path = "./examples/output/" # Path to folder in which to save output

In [7]:
# Recursive function to iterate over all files and remove high frequency traces
def process_directory(input_path,output_path,fail_list,stats=None,skip_existing=False):
    if not os.path.exists(output_path):
        os.makedirs(output_path)
    
    for file_name in os.listdir(input_path):
        file_path_in = os.path.join(input_path,file_name)
        
        if os.path.isdir(file_path_in):
            process_directory(file_path_in, os.path.join(output_path,file_name), fail_list, stats=stats, skip_existing=skip_existing)
            continue

        print(f'Processing {file_path_in}')

        if file_name == ".DS_Store":
            print('...Skipping')
            print('')
            continue
        
        file_path_out = os.path.join(output_path,file_name)

        if os.path.exists(file_path_out):
            print('...Exists (skipping)')
            continue
        
        if not file_name.endswith('.h5'):
            print('...Copying')
            shutil.copyfile(file_path_in, file_path_out)
        else:
            print('...Shrinking')
            try:
                file = pylake.File(file_path_in)
                file.save_as(file_path_out, omit_data={"Force HF/*"}, verbose=False)
            except:
                print('...FAILED (Copying instead)')
                fail_list.append(file_path_in)
                shutil.copyfile(file_path_in, file_path_out)
            
        update_stats(stats, file_path_in, file_path_out)
        print('')

# Stores rolling file size totals and file counts
def update_stats(stats, file_path_in, file_path_out):
    if stats is not None:
        stats[0] = stats[0] + os.path.getsize(file_path_in)
        stats[1] = stats[1] + os.path.getsize(file_path_out)
        stats[2] = stats[2] + 1
    
        print_stats(stats)

# Displays current filesize reduction statistics
def print_stats(stats):
    print('...Processed %i files, reducing %.2f MB to %.2f MB (%.2f%%)' % (stats[2],stats[0]/1048576,stats[1]/1048576,stats[1]/stats[0]*100))


In [8]:
# Starting processing at root directory.  Providing empty list to store any failed files
fail_list = []
process_directory(input_path, output_path, fail_list, stats=[0,0,0], skip_existing=True)

if len(fail_list) > 0:
    print('Failed for the following files')
    print(fail_list)

Processing ./examples/input/20230324-155241 Scan 42.h5
...Exists (skipping)
